In [1]:
import pandas as pd
import numpy as np
import re

print("Starting Data Preprocessing Pipeline...")

# STEP 1: LOAD RAW DATA
# Using relative paths: go up one folder (../), then into data/raw/
raw_file_path = "../data/raw/Hela_Apparel_Master_Data.csv"

try:
    df = pd.read_csv(raw_file_path)
    print(f"Loaded Raw Data successfully. Total rows: {len(df)}")
except FileNotFoundError:
    print("ERROR: Could not find the file. Make sure it is inside the 'data/raw/' folder!")
    raise

Starting Data Preprocessing Pipeline...
Loaded Raw Data successfully. Total rows: 5026


In [2]:
# STEP 1.5: INITIAL DATA EXPLORATION & MISSING VALUE AUDIT
print("Dataset Structural Audit")
# df.info() gives the data types and total non-null counts
df.info()

print("\n --- Missing Values Detected ---")
# calculate the sum of NaNs in every column
missing_data = df.isnull().sum()

# filter to ONLY show columns that actually have missing data
missing_columns = missing_data[missing_data > 0]

if not missing_columns.empty:
    print(missing_columns)
    # print("\n Conclusion: Key operational and demographic columns (District, Race, Team) contain missing values. Active employees naturally lack a 'Last Working Date'. Automated mode-imputation is required for the demographic features to prevent dropping 25% of the dataset.")
else:
    print("No missing values found.")

Dataset Structural Audit
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5026 entries, 0 to 5025
Data columns (total 17 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   EMP No               5026 non-null   int64  
 1   Name                 5026 non-null   object 
 2   Level                5026 non-null   object 
 3   Designation          5026 non-null   object 
 4   Grade                5026 non-null   object 
 5   Team                 5024 non-null   object 
 6   Date Of Appointment  5026 non-null   object 
 7   Service Year(s)      5026 non-null   object 
 8   Age                  5026 non-null   object 
 9   Last Working Date    3526 non-null   object 
 10  Reason               3526 non-null   object 
 11  Unnamed: 11          0 non-null      float64
 12  Race                 5023 non-null   object 
 13  Gender               5026 non-null   object 
 14  Marial Status        5026 non-null   object 
 15  Transport mod

In [3]:
# STEP 2: FEATURE EXTRACTION (Text to Math)
def parse_duration(duration_str):
    """Converts strings like '1Y, 5M, 11D' into a decimal like 1.41"""
    if pd.isna(duration_str): return 0.0
    if isinstance(duration_str, (int, float)): return float(duration_str)
    
    # Extract years and months using Regular Expressions (Regex)
    y_match = re.search(r'(\d+)Y', str(duration_str))
    m_match = re.search(r'(\d+)M', str(duration_str))
    
    years = int(y_match.group(1)) if y_match else 0
    months = int(m_match.group(1)) if m_match else 0
    
    # Calculate exact decimal
    return round(years + (months / 12), 2)

df['Service_Years_Numeric'] = df['Service Year(s)'].apply(parse_duration)

# Extract integer years from string duration formats
df['Age_Numeric'] = df['Age'].apply(parse_duration) 

# Calculate age at the exact time of joining
df['Age_At_Hiring'] = df['Age_Numeric'] - df['Service_Years_Numeric']

# Drop the mathematically biased column entirely
df = df.drop(columns=['Age_Numeric'])
print("Converted 'Age' and 'Service Years' text to numeric features.")

Converted 'Age' and 'Service Years' text to numeric features.


In [4]:
# STEP 3: CREATE THE TARGET VARIABLE (Infant Attrition)
def determine_target(row):
    """
    Target = 1.0 (Failed): Left the company in < 1 year.
    Target = 0.0 (Success): Reached the 1-year mark (tenure >= 1.0).
    Target = NaN (Unknown): Active employee with < 1 year tenure.
    """
    is_leaver = pd.notna(row['Last Working Date'])
    tenure = row['Service_Years_Numeric']
    
    if is_leaver and tenure < 1.0:
        return 1.0  # Infant Attrition (Definite Failure)
    elif tenure >= 1.0:
        return 0.0  # Survived 12 months (Definite Success)
    else:
        return np.nan # Active but < 1 year (Unknown Outcome)

# 1. Apply the function
df['Early_Attrition'] = df.apply(determine_target, axis=1)

# 2. Count the unknowns before dropping
unknown_count = df['Early_Attrition'].isna().sum()
print(f"Identified {unknown_count} active employees with < 1 year tenure.")
print("Dropping these 'Unknown' records to prevent right-censored bias in training...")

# 3. Drop the rows where the outcome is still unknown
df = df.dropna(subset=['Early_Attrition']).copy()

# 4. Convert the target back to clean integers (0 and 1)
df['Early_Attrition'] = df['Early_Attrition'].astype(int)

print("\n Final Target Variable Balance:")
print(df['Early_Attrition'].value_counts(normalize=True) * 100)

Identified 100 active employees with < 1 year tenure.
Dropping these 'Unknown' records to prevent right-censored bias in training...

 Final Target Variable Balance:
Early_Attrition
1    50.0203
0    49.9797
Name: proportion, dtype: float64


In [6]:
# STEP 4: HANDLE MISSING DEMOGRAPHICS (Imputation)
cols_to_clean = ['District', 'Transport mode', 'Marial Status', 'Race', 'Gender']
for col in cols_to_clean:
    if col in df.columns and df[col].isnull().sum() > 0:
        # Fill missing blanks with the most frequent value (the Mode)
        mode_val = df[col].mode()[0]
        df[col] = df[col].fillna(mode_val)

print("Handled missing demographic values using Mode Imputation.")

Handled missing demographic values using Mode Imputation.


In [7]:
# STEP 5: PREVENT DATA LEAKAGE & CLEANUP
# Drop columns that give away the answer, plus old text columns
columns_to_drop = [
    'Last Working Date', 'Reason', 'Service Year(s)', 'Age', 
    'Date Of Appointment', 'Name', 'EMP No'
]
if 'Unnamed: 11' in df.columns:
    columns_to_drop.append('Unnamed: 11')

# Only drop columns that actually exist in the dataframe to prevent errors
existing_cols_to_drop = [col for col in columns_to_drop if col in df.columns]
df_clean = df.drop(columns=existing_cols_to_drop)

In [9]:
# STEP 6: SAVE TO PROCESSED FOLDER
processed_file_path = "../data/processed/Hela_ML_Ready.csv"
df_clean.to_csv(processed_file_path, index=False)

print(f"\n SUCCESS! Clean data saved to: {processed_file_path}")
print("Ready for Exploratory Data Analysis (EDA)!")


 SUCCESS! Clean data saved to: ../data/processed/Hela_ML_Ready.csv
Ready for Exploratory Data Analysis (EDA)!
